In [ ]:
#Identifying best positions for sextupoles
'''
def SextupolePosition(ring):
   tw = ring.twiss4d()

   sf_candidates = []
   sd_candidates = []
   
   sf_top = []
   sd_top = []

   for name in tw.name:
         row = tw.rows[name]
         dx = abs(row.dx[0])
         betx = row.betx[0]
         bety = row.bety[0]

         if dx > 0.3:
            if betx > bety:
               sf_candidates.append((name, dx, betx))
            else:
               sd_candidates.append((name, dx, bety))
   
   sf_candidates = sorted(sf_candidates, key=lambda x: x[1]*x[2], reverse=True)
   sd_candidates = sorted(sd_candidates, key=lambda x: x[1]*x[2], reverse=True)

   if sf_candidates:
      sf_top = sf_candidates[:10]
      for c in sf_top:
         print("SF_positions", c)

   if sd_candidates:
      sd_top = sd_candidates[:10]
      for c in sd_top:
         print("SD_positions", c)

   return sf_top, sd_top
   '''

In [ ]:
#Placing identified sextupoles

'''

for name _,_, in sf_candidates[:4]:
    ring.insert(pdr.new(f'SF_auto_{name},'SFarc'), at=name)

for name _,_, in sd_candidates[:4]:
    ring.insert(pdr.new(f'SD_auto_{name}, 'SDarc'), at=name)

    
'''

In [ ]:
#Chromaticity correction
'''
def ChromCorrect(dqx, dqy, MakePlot=False):
        
    opt_chrom = ring.match(
        solve=False,
        method='4d',
        vary=xt.VaryList([dqx, dqy], step=1e-3),
        targets=xt.TargetSet(dqx=0, dqy=0, tol=1e-3))
    opt_chrom.target_status()
    opt_chrom.run_jacobian(n_steps=100)
    opt_chrom.target_status()
    # Print the final matched strengths

    print(f"Matched kSF: {pdr.vars['kSF']._get_value():.6f}")
    print(f"Matched kSD: {pdr.vars['kSD']._get_value():.6f}")   
'''


In [ ]:
#Suitability of working point

'''
qx=tw_final.qx %1
qy=tw_final.qy %1

def plot_working_point(qx, qy, max_order=4):
    fig, ax = plt.subplots(figsize=(8, 8))
    drawn_lines = set()

    for order in range(1, max_order + 1):
        linewidth = 2.0 / order
        alpha = 1.0 / order
        
        for m in range(order + 1):
            n = order - m
            for p in range(-max_order, max_order * 2):
                
                common = math.gcd(m, n, p)
                if common == 0: continue 
                
                line_id = (m // common, n // common, p // common)
                
                if line_id not in drawn_lines:
                    if n != 0:
                        x = np.array([0, 1])
                        y = (p - m * x) / n
                        if np.any((y >= 0) & (y <= 1)) or np.any((y <= 0) & (y >= 1)):
                            ax.plot(x, y, 'k-', lw=linewidth, alpha=alpha)
                    elif m != 0:
                        x_pos = p / m
                        if 0 <= x_pos <= 1:
                            ax.axvline(x_pos, color='k', lw=linewidth, alpha=alpha)
                    
                    drawn_lines.add(line_id)

    # Plot the specific working point
    ax.plot(qx % 1, qy % 1, 'ro', ms=10, label=f'WP ({qx:.4f}, {qy:.4f})')
    
    # Formatting
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f'Tune Diagram - Order 1 to {max_order}')
    ax.set_xlabel('$Q_x$ fraction')
    ax.set_ylabel('$Q_y$ fraction')
    ax.grid(True, which='both', linestyle=':', alpha=0.5)
    ax.legend()
    
    plt.show()
plot_working_point(qx,qy,max_order=5)
'''